## 

# 2 Transformer 核心模块
## 2.1 一批 Token 是如何流过模型的？
![](/img/pytorch/transformer.png)
可以先把整个语言模型看成这样一条流水线：
```
Token IDs [B, S]
   ↓
Embedding
   ↓
Hidden States [B, S, d_model]
   ↓
N 个 Transformer Block
   ↓
Final Norm
   ↓
LM Head
   ↓
Logits [B, S, vocab_size]
   ↓
Softmax（训练时通常不显式写）
   ↓
每个位置预测“下一个 Token”的概率分布
```
- B：Batch Size，一次处理多少条样本
- S：Sequence Length，每条样本有多少个 Token
- d_model：模型主隐藏维度
- vocab_size：词表大小
- h：注意力头数
- d_head = d_model / h：每个注意力头的维度

### 2.1.1 输入阶段
模型最开始拿到的，并不是字符串，而是 Tokenizer 编码后的整数张量：`token_ids.shape == [B, S]`
```
[
  [332, 545, 612],
  [866, 959, 1402]
]
```
> 上面有2条样本，Batchsize=2；每条样本长度为3，所以SequenceLength=3

这些整数本身没有“数学语义”，Token ID 是离散符号，不是连续特征。
而神经网络的核心操作是向量加法、矩阵乘法、点积、非线性变换，所以模型必须先把离散的 ID 变成连续向量。这就是 Embedding 层存在的原因。

### 2.1.2 Embedding
Embedding 的本质可以概括为一个巨大的 **“查找表”**。
- 输入：`[B, S]`
- 输出：`[B, S, d_model]`
这里BPE Tokenizer输出的token ids每一个都对应一行参数，这个参数就是长度为`d_model`的浮点向量。
给定一个token id，就是去表里查“取第几行”。这些特征向量就可以去做**线性变换、注意力计算与非线性加工**。
**训练完成后，语义接近的token往往会在向量空间中更接近。**当然训练前肯定不是。
> 比如一个token id=545，输出就是一个d_model维度的向量比如说是4096

### 2.1.3 Transformer Block
一个Block通常由两个核心子模块组成：
1. self-attention：让不同位置间交换信息（看上下文）
2. FFN：让每个位置加工自己的特征（消化自己的信息）
#### 自注意力
在自注意力力，每个序列中的位置都在问：对我当前这个位置来说，前面哪些位置最重要？
模型会对每个位置的向量做三次线性投影，得到三种角色：
- Q（Query）：我想找什么
- K（Key）：我这里提供什么索引特征
- V（Value）：我这里真正携带的信息内容

输入`X.shape == [B, S, d_model]`，得到`Q=XW_Q、K=XW_K、V=XW_K`，形状都是`[B, S, d_model]`，都是**完全独立随机初始化**的。
对于某个位置t，我们拿它的Q_t去和所有位置的K_t做点积，点积越大，说明当前位置更应该关注那个位置。
最后得到**分数矩阵**：
$$scores = \frac{QK^\top}{\sqrt{d}}$$
> 这里的d是单头维度，即d_model/num_head，目的是为了数值稳定，避免点积随着维度变大而过大

得到**分数**后，对分数做Softmax得到0~1的概率，得到一组注意力权重，确定「我该重点关注谁」

用这组权重，也就是**相关度分数**去加权求和所有Value：
$$\text{Attention}(Q,K,V) = \text{softmax}\left( \frac{QK^\top}{\sqrt{d_k}} \right)V$$
最终才会融合真正的**特征信息**。由此一来，当前位置从整个上下文中，按需取回一份“上下文摘要”。

#### 多头注意力
单头注意力可以理解为用一种方式看上下文；
多头注意力则是用多种不同视角并行看上下文。例如某些头可能更擅长：
- 关注最近的词
- 关注主谓关系
- 关注标点或边界
- 关注长距离依赖
- 关注格式模式
因此，多头机制允许模型在多个子空间中并行建模。
> 为了保证这一点，所有小头的权重，都是 **独立随机初始化**


首先我们有一个`d_model = num_head * d_head`。把`Q/K/V`最后一维拆成多个头：`[B, S, num_head, d_head]`；
之后再**转置**成`[B, num_head, S, d_head]`，因为计算注意力时，`Q/K/V`**按head维度并行更方便**。
`Q × K_T` 后得到的**分数矩阵**形状是`[B, num_head, S, S] = [B, h, S, d_head] x [B, h, d_head, S]`（只是后面矩阵相乘(S, d_head) × (d_head, S) = (S, S)）
也就是每个批次、每个头、第 i 个词 对 所有 S 个词的相关性打分
- 第一个 S：当前词
- 第二个 S：所有要对比的词

分数矩阵左乘以矩阵V：`[B, num_head, S, S] x [B, num_head, S, d_head] = [B, num_head, S, d_head]`
也就是每个单头的注意力权重就是这个形状，再转置回去`[B, S, num_head, d_head]`并reshape成`[B, S, d_model]`
> 最后通常还会再经过一个输出投影层 W_O，输出 shape 保持不变

#### 因果编码
语言模型的训练目标是 **用前面的 token 去预测后面的 token**，所以第 t 个位置在计算时，只能看自己和自己之前的位置，不能偷看未来。否则训练就作弊了。

在注意力分数矩阵`[S, S]`上：
- 主对角线以上：小Q对大K，表示未来位置，不准看
- 主对角线及以下：OK

| 矩阵维度 | 名称 | 对应 | 含义 |
|---------|------|------|------|
| **行 (i)** | Query 位置 | Q | **当前正在看的词**（我是谁） |
| **列 (j)** | Key 位置 | K | **我能看的词**（看谁） |

```

          j=0   j=1   j=2   j=3
        +-----------------------+
i=0(词1)| ①     ②     ③     ④  |  ← 词1居然能看到词2/3/4（未来！违法）
i=1(词2)| ⑤     ⑥     ⑦     ⑧  |  ← 词2能看到词3/4（未来！违法）
i=2(词3)| ⑨     ⑩    ⑪    ⑫  |  ← 词3能看到词4（未来！违法）
i=3(词4)| ⑬    ⑭    ⑮    ⑯  |  ← 词4看全部（合法）
        +-----------------------+


          j=0   j=1      j=2        j=3
        +-----------------------------------+
i=0(词1)| ①    -∞      -∞        -∞     |  只看自己
i=1(词2)| ⑤     ⑥      -∞        -∞     |  看自己+前一个
i=2(词3)| ⑨     ⑩      ⑪        -∞     |  看前两个+自己
i=3(词4)| ⑬    ⑭      ⑮        ⑯     |  看全部
        +-----------------------------------+
```

总的来说，因果编码的目的就是为了满足 **自回归语言模型的因果约束**。
#### RoPE: Rotary Position Embedding
如果没有任何位置机制，那么自注意力只会知道来了一堆token向量，却不知道谁在前谁在后，或两个token相隔多远。
RoPE的思路是在Q和K上施加与位置相关的**旋转变换**。
**传统绝对位置编码（正弦/可学习PE）**：
$$X_{pos} = X + P$$
然后用这个带位置的 $X$ 去算：$Q=XW_q,\,K=XW_k,\,V=XW_v$
然而绝对位置编码它有3个致命硬伤
1. **位置污染了V（完全没必要）**
注意力逻辑本来是：
- $Q,K$：负责算「谁和谁有关系」→ 需要位置
- $V$：负责存「纯内容特征」→ **根本不需要位置**
直接加输入 = 强行把位置塞进了 $V$，多余、干扰语义。
2. 只能学**绝对位置**，不懂「相对距离」
比如：
- 句子第1、2个词（间隔1）
- 句子第100、101个词（间隔1）
老式PE：两个间隔1的词，相似度算出来**不一样**（因为绝对位置不同）
但人类语言里：**相邻词的亲密关系，和它在句子开头/结尾无关**！
3. 长文本外推拉胯
训练只见过短位置（比如≤2048），推理超长文本（4096/8192）时，陌生的绝对位置直接崩，模型乱关联。

与此同时，RoPE 核心操作不改输入、不加向量；只在注意力打分前，给每个位置的 $Q,K$ 做**角度旋转**：
- 位置 $m$ 的 $Q_m$ 转一个角度
- 位置 $n$ 的 $K_n$ 转一个角度
然后再算：$\boldsymbol{Q_m \cdot K_n}$
1. **只影响打分，不污染内容**
只动 $Q,K$（相似度计算环节），**完全不动V**！
位置信息只用来判断「该关注谁」，不篡改原本的语义特征，分工极致干净。
2. 天然自带**相对位置感知**
数学上可以推出来：
$Q_m \cdot K_n$ 的结果，**只和两个token的间隔 $(m-n)$ 有关**，和它们在句子里的绝对位置无关！
不管你在开头、中间、结尾，只要俩词隔3个位，关联强度永远一致——贴合人类语言逻辑。
3. 天生支持长文本外推
旋转是连续三角函数，没固定位置上限；LLaMA、Qwen全系大模型标配RoPE，就是因为能无缝撑超长上下文。

> 具体的代码见后面

#### FFN / SwiGLU
Attention 之后，每个位置已经把上下文信息读进来了，接下来模型还需要对每个位置自己的表示做进一步加工，这就是 **前馈神经网络** 的作用。
其基本操作就是升维、激活再降维：`Linear(d_model->d_ff) ——> 激活函数 ——> Linear(d_ff->d_model)`
对应过来的shape变化就是`[B, S, d_model] ——> [B, S, d_ff] ——> [B, S, d_model]`
$$FFN(x) = ReLU(x\cdot W_{up}) \cdot W_{down}$$

更大的中间维度，意味着：
- 更强的表达能力
- 更多的特征组合空间
- 更丰富的非线性加工能力

当然，传统ReLU好坏特征全混在一起，没用的噪音也留着，有点难绷，所以这里我们都是用门控机制（话说NIPS2025就有一篇说Qwen的门控注意力的？）
像这里绝大多数开源大模型用到的都是`SwiGLU = Swish + GLU`
- $$\sigma(x) = \frac{1}{1+e^{-x}}$$ 即 Sigmoid 函数
- $$\text{Swish}(x) = x \cdot \sigma(x)$$，如此以来 Swish 全程连续可导；负数不直接置 0，是平滑衰减

接着复习一下 GLU：
输入 $x$，拆分两个独立线性投影：
$$
\begin{align}
A &= x W_A \\
B &= x W_B
\end{align}
$$
$W_A,W_B$：可学习权重矩阵
$\odot$：**逐元素相乘（Hadamard Product）**
则 $$\text{GLU}(x) = \sigma(A) \odot B$$
- $\sigma(A)$：生成 0~1 的门控权重
- $B$：原始特征通路
- 相乘：门控过滤特征

最后 SwiGLU 顾名思义，就是把 GLU 的 Sigmoid 门换成 Swish：
$$
\text{SwiGLU}_\text{core}(x) = \text{Swish}(xW_A) \odot xW_B
$$

### 2.1.4 Block 输出
经过多个 Transformer Block 后，输出 shape `[B, S, d_model]`。但是我一直有一个疑问：**凭啥 Transformer Block 叠几层，向量就「语义更高级」？**
它全程无人类语义标注，只有一条数学任务：**给前 t 个词，猜第 t+1 个词（预测下一个 token）**
答案是——这是训练出来的，梯度惩罚。
比如说“苹果” 的向量，必须能精准算出下一个大概率是 “手机 / 水果”。
光靠原始词向量（刚查表出来的）根本做不到，因为单个 token 向量是孤立的，不知道上下文
那只能靠 Self-Attention 把整句话所有词的向量，加权揉在一起
- “我 吃 苹果” → 让「苹果」偷偷吸收「吃」的动作信息
- “我 用 苹果” → 让「苹果」吸收「用」的工具信息

就像多层卷积一样，叠多层 Transformer Block后：
- 第一层：学局部搭配（吃 + 苹果）
- 第二层：学短句逻辑（我吃苹果→常识）
- 第三层：学长距离关联（前文指代、逻辑因果）

既然聊到梯度了，就不得不提 **叠几十上百层** 的深度神经网络里非常可怕的现象：
- 某一层数值偏大 → 放大传到下一层 → 越传越大 → 梯度爆炸
- 某一层数值全挤很小 → 梯度趋近 0 → 完全学不动

#### LayerNorm
第一个归一化的方法是 LayerNorm，也就是对单个样本，单个向量，在最后一维（特征维）算这一条向量的均值与方差，将其强行标准化拉到均值0、方差1：
$$\hat x_i = \frac{x_i - \mu}{\sqrt{\sigma^2+\epsilon}}$$
> epsilon是一个超级小的正数比如 10^-6，防止方差为0
再用可学习缩放偏移：$y_i = \gamma \hat x_i + \beta$
- gamma 是模型自己训练更新的参数，不是固定数，把归一化后方差 = 1 的向量，按需放大（重要特征） / 缩小（无用特征）

不过这样因为算均值有减法，计算量方面比较大。特地摘出来说的原因就是下面的RMSNorm不算减法。

#### RMSNorm
